# Brownian Motion: A Short Numerical Analysis

This notebook provides a concise, reproducible analysis of **standard Brownian motion** and **geometric Brownian motion (GBM)** as implemented in this repository. We simulate both processes, verify their theoretical properties, and interpret the results in a quantitative finance context.

## 1. Standard Brownian Motion

**Definition.** Standard (Wiener) Brownian motion \(W_t\) is a continuous-time stochastic process with:
- \(W_0 = 0\)
- Independent increments: \(W_t - W_s\) independent of \(\mathcal{F}_s\) for \(t > s\)
- Stationary Gaussian increments: \(W_t - W_s \sim \mathcal{N}(0, t-s)\)

Hence \(W_T \sim \mathcal{N}(0, T)\). The simulation uses the discrete-time analogue: we take \(N\) steps of size \(\Delta t = T/N\), with increments \(\Delta W_k = \sqrt{\Delta t}\, Z_k\) where \(Z_k \sim \mathcal{N}(0,1)\) i.i.d., and set \(W_{t_n} = \sum_{k=1}^{n} \Delta W_k\) (with \(W_0 = 0\)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')
from brownian_motion import simulate_brownian_motion
from geometric_brownian_motion import simulate_geometric_brownian_motion

**Code note.** In `brownian_motion.py`, the core steps are:
1. Set \(\Delta t = T/N\) and draw \(\texttt{paths} \times N\) standard normals.
2. Scale by \(\sqrt{\Delta t}\) so \(\texttt{dW}\) has variance \(\Delta t\) per step (correct Brownian scaling).
3. Form paths by cumulative sum along the time axis and prepend a column of zeros so \(W(0)=0\).
4. Return the time grid \(t_0,\ldots,t_N\) and the matrix of paths \(W[i,n]\) for path \(i\) at time \(t_n\).

In [ ]:
# Parameters: horizon T, N steps, many paths for distributional checks
T_bm = 1.0
N_bm = 1000
paths_bm = 10000
t_bm, W_bm = simulate_brownian_motion(T=T_bm, N=N_bm, paths=paths_bm, seed=42)

W_T = W_bm[:, -1]
empirical_mean_bm = np.mean(W_T)
empirical_var_bm = np.var(W_T)

print("Standard Brownian motion — W(T)")
print("  Theoretical E[W(T)] = 0,    Var(W(T)) = T = ", T_bm)
print("  Empirical   mean    = {:.6f}, variance = {:.6f}".format(empirical_mean_bm, empirical_var_bm))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: sample paths
axes[0].plot(t_bm, W_bm[:5].T)
axes[0].set_title("Sample paths \(W_t\)")
axes[0].set_xlabel("Time \(t\)")
axes[0].set_ylabel("\(W(t)\)")

# Right: distribution of W(T) vs N(0,T)
axes[1].hist(W_T, bins=50, density=True, alpha=0.6, label="Simulated")
x = np.linspace(W_T.min(), W_T.max(), 300)
theory = (1 / np.sqrt(2 * np.pi * T_bm)) * np.exp(-x**2 / (2 * T_bm))
axes[1].plot(x, theory, label="\(\mathcal{N}(0,T)\)")
axes[1].set_title("Distribution of \(W(T)\)")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Density")
axes[1].legend()
plt.tight_layout()
plt.show()

**Interpretation.** The sample mean of \(W(T)\) is close to zero and the sample variance is close to \(T\), and the histogram aligns with the \(\mathcal{N}(0,T)\) density. This confirms that the discretisation and random number generation correctly approximate standard Brownian motion at terminal time.

## 2. Geometric Brownian Motion

**Definition.** GBM models an asset price \(S_t\) via the SDE
\[ dS_t = \mu S_t\,dt + \sigma S_t\,dW_t \]
with \(S_0 > 0\), drift \(\mu\) and volatility \(\sigma\). The solution is
\[ S_t = S_0 \exp\left( \big(\mu - \tfrac{\sigma^2}{2}\big) t + \sigma W_t \right). \]
Thus \(\ln(S_T/S_0)\) is Gaussian with mean \((\mu - \sigma^2/2)T\) and variance \(\sigma^2 T\), so \(S_T\) is log-normal.

**Code note.** In `geometric_brownian_motion.py`:
1. The same Brownian increments \(\Delta W\) are generated (scale \(\sqrt{\Delta t}\)).
2. Cumulative \(W\) is built as for standard BM, then the time grid \(t\) is used to form the exponent \((\mu - \sigma^2/2)t + \sigma W_t\) at each \((t, W)\).
3. The asset paths are \(S_t = S_0 \exp(\ldots)\). No Euler scheme is needed because we use the closed-form solution, which is exact in the discrete grid up to the approximation of \(W_t\).

In [ ]:
S0, mu, sigma = 100.0, 0.05, 0.2
T_gbm, N_gbm, paths_gbm = 1.0, 1000, 10000
t_gbm, S_gbm = simulate_geometric_brownian_motion(
    S0=S0, mu=mu, sigma=sigma, T=T_gbm, N=N_gbm, paths=paths_gbm, seed=42
)

S_T = S_gbm[:, -1]
E_ST_theory = S0 * np.exp(mu * T_gbm)
Var_ST_theory = S0**2 * np.exp(2 * mu * T_gbm) * (np.exp(sigma**2 * T_gbm) - 1)

print("Geometric Brownian motion — S(T)")
print("  S0 = {}, μ = {}, σ = {}, T = {}".format(S0, mu, sigma, T_gbm))
print("  Theoretical E[S(T)] = {:.4f},  Var(S(T)) = {:.4f}".format(E_ST_theory, Var_ST_theory))
print("  Empirical   mean   = {:.4f},  variance = {:.4f}".format(np.mean(S_T), np.var(S_T)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: sample price paths
axes[0].plot(t_gbm, S_gbm[:5].T)
axes[0].axhline(S0, color="gray", linestyle="--", alpha=0.7)
axes[0].set_title("Sample paths \(S_t\) (GBM)")
axes[0].set_xlabel("Time \(t\)")
axes[0].set_ylabel("\(S(t)\)")

# Right: distribution of S(T) vs log-normal
axes[1].hist(S_T, bins=50, density=True, alpha=0.6, label="Simulated")
x = np.linspace(S_T.min(), S_T.max(), 300)
log_mean = np.log(S0) + (mu - 0.5*sigma**2) * T_gbm
log_var = sigma**2 * T_gbm
density = (1 / (x * np.sqrt(2*np.pi*log_var))) * np.exp(-(np.log(x)-log_mean)**2 / (2*log_var))
axes[1].plot(x, density, label="Theoretical (log-normal)")
axes[1].set_title("Distribution of \(S(T)\)")
axes[1].set_xlabel("\(S(T)\)")
axes[1].set_ylabel("Density")
axes[1].legend()
plt.tight_layout()
plt.show()

**Interpretation.** The empirical mean and variance of \(S(T)\) match the theoretical log-normal moments, and the histogram agrees with the theoretical density. GBM is the standard model for equity prices in the Black–Scholes world; here we have verified both the path dynamics and the terminal distribution implied by the closed-form solution.

## 3. Summary

- **Standard Brownian motion** is simulated by summing scaled i.i.d. Gaussian increments; \(W(T)\) is \(\mathcal{N}(0,T)\) and path-wise behaviour is consistent with the theory.
- **Geometric Brownian motion** is simulated using the exact solution \(S_t = S_0 \exp((\mu-\sigma^2/2)t + \sigma W_t)\), with the same underlying \(W_t\); \(S(T)\) is log-normal and the implemented code reproduces its mean and variance.

Both modules are suitable as building blocks for option pricing (e.g. Monte Carlo), risk metrics, or path-dependent payoff analysis.